In [16]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig

from peft import LoraConfig, get_peft_model, TaskType

In [3]:
import torch, sys
print("python:", sys.executable)
print("torch version:", torch.__version__)
print("torch file:", torch.__file__)
print("has set_submodule:", hasattr(torch.nn.Module, "set_submodule"))

python: /usr/bin/python
torch version: 2.10.0+cu128
torch file: /usr/local/lib/python3.11/dist-packages/torch/__init__.py
has set_submodule: True


In [10]:
import sys
print(sys.executable)

/usr/bin/python


In [4]:
model_name = 'TinyLLama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = 'auto',
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [6]:
##gsm8k maths dataset 
torch.cuda.memory_allocated() / 1e9

0.829425664

In [7]:
###rank - ie, #r = Rank dimension - typically between 4-32
##rank he LoRA rank (r) essentially controls the size of the weight update, but in a specific low-rank factorized form.
# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
##rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
##lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
##lora_dropout = 0.05

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [8]:
##first 200 examples of the training data
data = load_dataset('openai/gsm8k', 'main', split='train[:200]')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [9]:
def tokenize(batch):
    texts = [
        f"### Instruction:\n{inst}\n### Response:\n{out}"
        for inst, out in zip(batch['question'], batch['answer'])
    ]
    tokens = tokenizer(
        texts,
        padding = 'max_length',
        truncation = True,
        max_length = 256,
        return_tensors = 'pt'
    )
    tokens['labels'] = tokens['input_ids'].clone()

    return tokens

In [10]:
tokenized_data = data.map(tokenize, batched=True, remove_columns=data.column_names)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
###1e-3 - heaving learning 
###epochs - 50 - eg 50 complete passes through the entire training dataset.
###
training_args = TrainingArguments(
    output_dir = './tinyllama-lora-tuned',
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 1e-3,
    num_train_epochs = 50,
    fp16 = True,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,      
    label_names = ["labels"]
)

In [12]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_data,
    processing_class = tokenizer
)

In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
20,1.784274
40,0.863513
60,0.750309
80,0.671513
100,0.587156
120,0.526027
140,0.453683
160,0.366112
180,0.315425
200,0.249382


TrainOutput(global_step=650, training_loss=0.24558868151444654, metrics={'train_runtime': 934.998, 'train_samples_per_second': 10.695, 'train_steps_per_second': 0.695, 'total_flos': 1.590741172224e+16, 'train_loss': 0.24558868151444654, 'epoch': 50.0})

In [15]:
model.save_pretrained("./tinyllama-lora-tuned-adapter-math")
tokenizer.save_pretrained("./tinyllama-lora-tuned-adapter-math")

('./tinyllama-lora-tuned-adapter-math/tokenizer_config.json',
 './tinyllama-lora-tuned-adapter-math/chat_template.jinja',
 './tinyllama-lora-tuned-adapter-math/tokenizer.json')

In [17]:
import os
print(os.getcwd())

/
